In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# On Kaggle the competition data is mounted read-only; locally it lives in ./data
KAGGLE_DIR  = Path('/kaggle/input/open-polymer-challenge')
DATA_DIR    = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get('POLYMER_DATA_DIR', 'data'))
RESULTS_DIR = Path(os.environ.get('POLYMER_RESULTS_DIR', 'results'))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameter-search budget (the Kaggle submission used 10 trials x 200 epochs)
N_TRIALS = int(os.environ.get('N_TRIALS', 10))
EPOCHS   = int(os.environ.get('EPOCHS', 200))

for f in sorted(DATA_DIR.glob('*.csv')):
    print(f)
print(f'N_TRIALS={N_TRIALS}, EPOCHS={EPOCHS}')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = DATA_DIR / 'train.csv'
train_df = pd.read_csv(csv_path)

# 1. split off 20% for dev_test
temp_df, dev_test = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,  # for reproducibility
    shuffle=True
)

# 2. split the remaining 80% into 75% train / 25% valid → 0.6 / 0.2 overall
dev_train, dev_val = train_test_split(
    temp_df,
    test_size=0.25,  # 0.25 * 0.8 = 0.2 of the original
    random_state=42,
    shuffle=True
)

# Verify sizes
print(f"Total rows:   {len(train_df)}")
print(f"Dev train:    {len(dev_train)} ({len(dev_train)/len(train_df):.2%})")
print(f"Dev valid:    {len(dev_val)} ({len(dev_val)/len(train_df):.2%})")
print(f"Dev test:     {len(dev_test)} ({len(dev_test)/len(train_df):.2%})")
print(f"Polymer example:{dev_train['SMILES'].to_list()[:3]}")
print(f"Columns:{dev_train.columns}")

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

X_train = dev_train['SMILES'].to_list()
y_train = dev_train[['Tg', 'FFV', 'Tc', 'Density', 'Rg']].to_numpy()
X_val   = dev_val['SMILES'].to_list()
y_val   = dev_val[['Tg', 'FFV', 'Tc', 'Density', 'Rg']].to_numpy()
X_test  = dev_test['SMILES'].to_list()
y_test  = dev_test[['Tg', 'FFV', 'Tc', 'Density', 'Rg']].to_numpy()
task_names = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']

# Property ranges for the competition-style weighted MAE (taken from the full train.csv labels)
label_range = (train_df[task_names].max() - train_df[task_names].min()).to_numpy()
results = {}   # model name -> metrics dict, filled in by each evaluation cell

def wmae(y_true, y_pred):
    """Weighted MAE in the style of the competition metric: per-property MAE divided by the
    property's range, weighted by sqrt(1/n_labels) and normalised so the weights sum to 5."""
    maes, counts = [], []
    for i in range(len(task_names)):
        mask = ~np.isnan(y_true[:, i])
        maes.append(np.mean(np.abs(y_true[mask, i] - y_pred[mask, i])) / label_range[i])
        counts.append(mask.sum())
    w = np.sqrt(1 / np.array(counts))
    w = w / w.sum() * len(w)
    return float(np.average(maes, weights=w))

def compute_mse(y_true, y_pred, name=None):
    print("MSE per task:")
    row = {}
    for i, task in enumerate(task_names):
        mask = ~np.isnan(y_true[:, i])
        if mask.sum() > 0:
            mse = mean_squared_error(y_true[mask, i], y_pred[mask, i])
            row[f'MSE {task}'] = mse
            row[f'MAE {task}'] = mean_absolute_error(y_true[mask, i], y_pred[mask, i])
            print(f"  {task}: {mse:.4f}")
        else:
            print(f"  {task}: N/A")
    mask_all = ~np.isnan(y_true)
    overall = mean_squared_error(y_true[mask_all], y_pred[mask_all])
    row['Overall MSE'] = overall
    row['wMAE'] = wmae(y_true, y_pred)
    print(f"Overall MSE: {overall:.4f}")
    print(f"Weighted MAE (competition-style): {row['wMAE']:.4f}")
    if name is not None:
        results[name] = row
    return row

In [ ]:
import time
from torch_molecule import LSTMMolecularPredictor
from torch_molecule.utils.search import ParameterType, ParameterSpec

search_parameters = {
    "output_dim": ParameterSpec(ParameterType.INTEGER, (8, 32)),
    "LSTMunits": ParameterSpec(ParameterType.INTEGER, (30, 120)),
    "learning_rate": ParameterSpec(ParameterType.LOG_FLOAT, (1e-4, 1e-2)),
}

lstm = LSTMMolecularPredictor(
    task_type="regression",
    num_task=5,
    batch_size=192,
    epochs=EPOCHS,
    verbose="none",
)

print("Model initialized successfully")
t0 = time.time()
lstm.autofit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    search_parameters=search_parameters,
    n_trials=N_TRIALS
)
print(f"Search finished in {(time.time() - t0) / 60:.1f} min")
print("Best hyperparameters:", {k: lstm.get_params()[k] for k in search_parameters})
print("Best epoch:", lstm.fitting_epoch + 1)

In [ ]:
y_predict_lstm = lstm.predict(X_test)['prediction']
compute_mse(y_test, y_predict_lstm, name='LSTM')

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor

# Morgan fingerprint, radius 2, 1024 bits (same bits as AllChem.GetMorganFingerprintAsBitVect)
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return morgan.GetFingerprintAsNumPy(mol)

X_train_feats = np.vstack([smiles_to_fp(s) for s in X_train])
X_val_feats   = np.vstack([smiles_to_fp(s) for s in X_val])
X_test_feats  = np.vstack([smiles_to_fp(s) for s in X_test])

X_dev_feats = np.vstack([X_train_feats, X_val_feats])
y_dev = np.vstack([y_train, y_val])

models = {}
y_pred_rf = np.zeros_like(y_test)

for idx, name in enumerate(task_names):
    print('Training random forest for:', name)
    y_col = y_dev[:, idx]
    mask  = ~np.isnan(y_col)
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_dev_feats[mask], y_col[mask])
    models[name] = rf
    y_pred_rf[:, idx] = rf.predict(X_test_feats)

compute_mse(y_test, y_pred_rf, name='Random Forest')

In [ ]:
from torch_molecule import GNNMolecularPredictor
from torch_molecule.utils.search import ParameterType, ParameterSpec

search_parameters = {
    'num_layer': ParameterSpec(param_type=ParameterType.INTEGER, value_range=(2, 5)),
    'hidden_size': ParameterSpec(param_type=ParameterType.INTEGER, value_range=(64, 512)),
    'learning_rate': ParameterSpec(param_type=ParameterType.LOG_FLOAT, value_range=(1e-4, 1e-2)),
}

gnn = GNNMolecularPredictor(
    task_type="regression",
    num_task=5,
    batch_size=192,
    epochs=EPOCHS,
    verbose="none",
)

print("Model initialized successfully")
t0 = time.time()
gnn.autofit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    search_parameters=search_parameters,
    n_trials=N_TRIALS
)
print(f"Search finished in {(time.time() - t0) / 60:.1f} min")
print("Best hyperparameters:", {k: gnn.get_params()[k] for k in search_parameters})
print("Best epoch:", gnn.fitting_epoch + 1)

In [ ]:
y_predict_gnn = gnn.predict(X_test)['prediction']
compute_mse(y_test, y_predict_gnn, name='GNN')

In [ ]:
# The submission averages LSTM and GNN; score that ensemble on the same dev-test split
y_predict_ens = (y_predict_lstm + y_predict_gnn) / 2
compute_mse(y_test, y_predict_ens, name='LSTM + GNN (submitted)')

## Dev-test results

All models are scored on the same held-out 20% split (1,595 polymers; only the labelled entries of each property count).
`wMAE` follows the competition metric: each property's MAE is divided by its range and weighted by `sqrt(1/n)`, so
the sparse properties (Tg, Tc, Density, Rg) count as much as FFV. Lower is better for every column.

In [ ]:
import matplotlib.pyplot as plt

summary_df = pd.DataFrame(results).T
summary_df.index.name = 'Model'
summary_df.to_csv(RESULTS_DIR / 'dev_test_metrics.csv')

pd.set_option('display.float_format', lambda v: f'{v:.4g}')
summary_df[['wMAE', 'Overall MSE'] + [f'MAE {t}' for t in task_names]]

In [ ]:
model_names = list(results)
colors = ['#2a6fdb', '#e8833a', '#3a9e6f', '#8b5cc7']
fig, axes = plt.subplots(1, len(task_names), figsize=(17, 3.6), sharey=False)
for ax, task in zip(axes, task_names):
    vals = [results[m][f'MAE {task}'] for m in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors[:len(model_names)], width=0.7)
    ax.bar_label(bars, fmt='%.3g', fontsize=8, padding=2)
    ax.set_title(f'{task}  (n = {int((~np.isnan(y_test[:, task_names.index(task)])).sum())})', fontsize=11)
    ax.set_xticks([])
    ax.spines[['top', 'right']].set_visible(False)
    ax.margins(y=0.15)
axes[0].set_ylabel('MAE on dev-test')
fig.legend(bars, model_names, loc='lower center', ncol=len(model_names), frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Per-property MAE on the held-out dev-test split (lower is better)', fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'dev_test_mae_by_property.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(sample_sub.head())

test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(test_df.head())

X_final = test_df['SMILES'].to_list()

lstm_preds = lstm.predict(X_final)['prediction']
gnn_preds  = gnn.predict(X_final)['prediction']
preds = (lstm_preds + gnn_preds) / 2

submission_df = sample_sub.copy()
submission_df[['Tg', 'FFV', 'Tc', 'Density', 'Rg']] = preds
print('submission_df', submission_df)

In [ ]:
# save to CSV (submission.csv in the working directory is what Kaggle picks up)
submission_df.to_csv('submission.csv', index=False)
submission_df.to_csv(RESULTS_DIR / 'submission.csv', index=False)